# DigiCow Farmer Adoption Prediction

**Competition:** DigiCow Farmer Adoption Prediction - Zindi Africa
**Goal:** Predict the probability that a farmer adopts an agricultural practice

within 7, 90, and 120 days of attending a training session, using only
information available at the time of training.

**Approach:** We build interpretable models that produce probability scores
for each prediction window. The focus throughout is on features and decisions
that can be explained to a non-technical audience.

---

### Data Available

| File | Description |
|---|---|
| `Train.csv` | Labelled training records: farmers with known adoption outcomes |
| `Test.csv` | Unlabelled records: farmers whose adoption we predict |
| `Prior.csv` | Historical training records from an earlier period: used for feature construction |

### Notebook Structure

| Section | Content |
|---|---|
| 1 | Data loading and sanity checks |
| 2 | Exploratory data analysis |
| 3 | Feature engineering |
| 4 | Logistic Regression models with odds ratios |
| 5 | Decision Tree models with visualisation |
| 6 | Classifier Chain - exploiting target structure |
| 7 | Model comparison |
| 8 | Interpretation |

### Libraries 

`pandas` is used throughout this notebook for data manipulation. All datasets are loaded and processed as `pandas` DataFrames.

`numpy` supports numerical operations used in feature engineering and metric calculations.

`matplotlib` is the base plotting library. All charts in this notebook are built using `matplotlib` figures and axis

`seaborn` sits on top of `matplotlib` and simplifies grouped bar charts and heatmaps. 

`os` is used to create the figures folder if it does not already exist.

`ast` is used to parse the topics column, which is stored as a string
representation of a Python list.

In [38]:
# ── Libraries ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


# ── Output folder ─────────────────────────────────────────────────────────────
# All charts are saved here so they can be referenced in the README.
os.makedirs('figures', exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
# Any function that involves randomness accepts a random_state parameter.
# Passing the same integer every time ensures results are identical across runs.
RANDOM_SEED = 42



# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_rows', 100)

print("Libraries loaded.")


Libraries loaded.


## 1. Data Loading and Initial Overview

We load all three files and verify they match expectations. This section establishes that the data is intact and that the relationship between the three files is what we expect.

The three files serve different purposes:
- `Train.csv` is used to train the models, it contains known outcomes
- `Test.csv` contains farmers whose adoption we need to predict
- `Prior.csv` contains historical records from an earlier period. We will use this in Section 3 to build features without leaking information about the outcomes we are predicting

In [39]:
train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')
prior_df = pd.read_csv('Prior.csv')

print("Files loaded.")
print(f"  Train : {train_df.shape[0]:,} rows  {train_df.shape[1]} columns")
print(f"  Test  : {test_df.shape[0]:,} rows  {test_df.shape[1]} columns")
print(f"  Prior : {prior_df.shape[0]:,} rows  {prior_df.shape[1]} columns")

Files loaded.
  Train : 13,536 rows  17 columns
  Test  : 5,621 rows  14 columns
  Prior : 44,882 rows  17 columns


In [40]:
print("=== TEST COLUMNS ===")
for col in test_df.columns:
    print(f"  {col}")
print(f"Shape: {test_df.shape}")

print("=== TRAIN COLUMNS ===")
for col in train_df.columns:
    print(f"  {col}")
print(f"Shape: {train_df.shape}")

print("\n=== PRIOR COLUMNS ===")
for col in prior_df.columns:
    print(f"  {col}")
print(f"Shape: {prior_df.shape}")

print("\n=== PRIOR SAMPLE ===")
print(prior_df.head(3).to_string())

=== TEST COLUMNS ===
  ID
  farmer_name
  training_day
  gender
  registration
  age
  group_name
  belong_to_cooperative
  county
  subcounty
  ward
  has_topic_trained_on
  trainer
  topics_list
Shape: (5621, 14)
=== TRAIN COLUMNS ===
  ID
  farmer_name
  training_day
  gender
  registration
  age
  group_name
  belong_to_cooperative
  county
  subcounty
  ward
  adopted_within_07_days
  adopted_within_90_days
  adopted_within_120_days
  has_topic_trained_on
  trainer
  topics_list
Shape: (13536, 17)

=== PRIOR COLUMNS ===
  ID
  farmer_name
  training_day
  gender
  registration
  age
  group_name
  belong_to_cooperative
  county
  subcounty
  ward
  adopted_within_07_days
  adopted_within_90_days
  adopted_within_120_days
  has_topic_trained_on
  trainer
  topics_list
Shape: (44882, 17)

=== PRIOR SAMPLE ===
          ID  farmer_name training_day  gender registration       age   group_name  belong_to_cooperative      county    subcounty          ward  adopted_within_07_days  adopte

### 1.3 Data Types and Missing Values

We inspect each dataset for two things:

(i) how each column is stored

(ii) whether any values are missing. 


In [41]:
print("=== Train ===")
train_df.info()

print("\n=== Test ===")
test_df.info()

print("\n=== Prior ===")
prior_df.info()

=== Train ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ID                       13536 non-null  object
 1   farmer_name              13536 non-null  object
 2   training_day             13536 non-null  object
 3   gender                   13536 non-null  object
 4   registration             13536 non-null  object
 5   age                      13536 non-null  object
 6   group_name               13536 non-null  object
 7   belong_to_cooperative    13536 non-null  int64 
 8   county                   13536 non-null  object
 9   subcounty                13536 non-null  object
 10  ward                     13536 non-null  object
 11  adopted_within_07_days   13536 non-null  int64 
 12  adopted_within_90_days   13536 non-null  int64 
 13  adopted_within_120_days  13536 non-null  int64 
 14  has_topic_trained_on    

In [42]:
# Checking for missing values in train and test datasets
print("=== Missing Values — Train ===")
missing = train_df.isnull().sum() # counts missing values per column
missing_pct = (missing / len(train_df) * 100).round(2) # Percentage of missing values per column
print(pd.DataFrame({'missing_count': missing,
                    'missing_pct': missing_pct})
      [missing > 0])

print("\n=== Missing Values — Test ===")
missing = test_df.isnull().sum()
missing_pct = (missing / len(test_df) * 100).round(2)
print(pd.DataFrame({'missing_count': missing,
                    'missing_pct': missing_pct})
      [missing > 0])

=== Missing Values — Train ===
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

=== Missing Values — Test ===
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


No missing values were found in either Train or Test. This means every row is complete and no imputation is needed before modelling.

In [43]:
# Check exact values in the registration column
print(train_df['registration'].value_counts())

# Check the topics_list — first 3 values
print()
for val in train_df['topics_list'].head(3):
    print(val)
    print()

# Check Prior adoption column name and a sample
print(prior_df[['trainer', 'county', 'adopted_within_120_days']].head(3))

registration
Ussd      8383
Manual    5153
Name: count, dtype: int64

[['Ndume App', 'Poultry Feeding']]

[['Poultry Housing'], ['Poultry Housing']]

[['Asili Fertilizer (Organic)', 'Biosecurity In Poultry Farming', 'Calf Feeding', 'Dairy Health Management', 'Dairy Nutrition With Tyari', 'Diseases In Dairy Farming', 'Feeding A Lactating Cow', 'How To Feed Kienyeji Chicken For Meat', 'How To Feed Kienyeji Chicken From 1 Day Old To Maturity', 'How To Feed Layers From 1 Day Old To Maturity', 'How To Rear A Calf With Unga Products', 'How To Rear Healthy Chicken With Biodeal.', 'Importance Of Choosing The Right Seed Variety', 'Importance Of Vaccinations And Record', 'Milking Hygiene', 'Personal Protective Equipment (Ppe)', 'Pest And Disease Management In Maize And Beans', 'Poultry And Dairy Feeding With Tyari Feeds', 'Poultry Diseases And Biosecurity', 'Poultry Feeding', 'Poultry Feeding With Tyari', 'Poultry Health Management', 'Poultry Housing', 'Poultry Products', 'Record Keeping In Dair

In [44]:
# ── Reusable summary function ─────────────────────────────────────────────────
# Defined once, called on all three DataFrames

def summarise_dataframe(df, name):
    """
    Returns a column-level audit table.
    
    Parameters
    ----------
    df   : pd.DataFrame
    name : str — label for the printed header
    """
    summary = pd.DataFrame({
        'dtype'         : df.dtypes,
        'missing_count' : df.isnull().sum(),
        'missing_pct'   : (df.isnull().sum() / len(df) * 100).round(2),
        'n_unique'      : df.nunique(),
        'has_missing'   : df.isnull().sum() > 0
    })

    print(f"\n{'═' * 60}")
    print(f"  {name}  |  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
    print(f"{'═' * 60}")
    return summary

train_summary = summarise_dataframe(train_df, "Train.csv")
display(train_summary)

test_summary = summarise_dataframe(test_df, "Test.csv")
display(test_summary)

prior_summary = summarise_dataframe(prior_df, "Prior.csv")
display(prior_summary)


════════════════════════════════════════════════════════════
  Train.csv  |  13,536 rows  ×  17 columns
════════════════════════════════════════════════════════════


,dtype,missing_count,missing_pct,n_unique,has_missing
ID,object,0,0.0000,13536,False
farmer_name,object,0,0.0000,13536,False
training_day,object,0,0.0000,152,False
gender,object,0,0.0000,2,False
registration,object,0,0.0000,2,False
age,object,0,0.0000,2,False
group_name,object,0,0.0000,864,False
belong_to_cooperative,int64,0,0.0000,2,False
county,object,0,0.0000,8,False
subcounty,object,0,0.0000,26,False



════════════════════════════════════════════════════════════
  Test.csv  |  5,621 rows  ×  14 columns
════════════════════════════════════════════════════════════


,dtype,missing_count,missing_pct,n_unique,has_missing
ID,object,0,0.0000,5621,False
farmer_name,object,0,0.0000,5621,False
training_day,object,0,0.0000,78,False
gender,object,0,0.0000,2,False
registration,object,0,0.0000,2,False
age,object,0,0.0000,2,False
group_name,object,0,0.0000,239,False
belong_to_cooperative,int64,0,0.0000,2,False
county,object,0,0.0000,7,False
subcounty,object,0,0.0000,16,False



════════════════════════════════════════════════════════════
  Prior.csv  |  44,882 rows  ×  17 columns
════════════════════════════════════════════════════════════


,dtype,missing_count,missing_pct,n_unique,has_missing
ID,object,0,0.0000,44882,False
farmer_name,object,0,0.0000,6719,False
training_day,object,0,0.0000,224,False
gender,object,0,0.0000,2,False
registration,object,0,0.0000,2,False
age,object,0,0.0000,2,False
group_name,object,0,0.0000,840,False
belong_to_cooperative,int64,0,0.0000,2,False
county,object,0,0.0000,9,False
subcounty,object,0,0.0000,28,False
